pip install openmeteo-requests
pip install requests-cache retry-requests numpy pandas

In [2]:

%pip install openmeteo-requests
%pip install requests-cache retry-requests numpy pandas

  Using cached openmeteo_requests-1.7.5-py3-none-any.whl.metadata (11 kB)
  Using cached niquests-3.17.0-py3-none-any.whl.metadata (17 kB)
  Using cached openmeteo_sdk-1.25.0-py3-none-any.whl.metadata (935 bytes)
  Using cached charset_normalizer-3.4.4-cp314-cp314-macosx_10_13_universal2.whl.metadata (37 kB)
  Using cached urllib3_future-2.15.902-py3-none-any.whl.metadata (16 kB)
  Using cached wassima-2.0.4-py3-none-any.whl.metadata (3.7 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached jh2-5.0.10-cp37-abi3-macosx_10_12_x86_64.macosx_11_0_arm64.macosx_10_12_universal2.whl.metadata (4.0 kB)
  Using cached qh3-1.5.6-cp37-abi3-macosx_10_12_x86_64.macosx_11_0_arm64.macosx_10_12_universal2.whl.metadata (4.8 kB)
  Using cached flatbuffers-25.9.23-py2.py3-none-any.whl.metadata (875 bytes)
Using cached openmeteo_requests-1.7.5-py3-none-any.whl (7.1 kB)
Using cached niquests-3.17.0-py3-none-any.whl (183 kB)
Using cached charset_normalizer-3.4.4-cp314-cp314-macosx_

In [1]:

import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": 52.52,
	"longitude": 13.41,
	"start_date": "2024-01-01",
	"end_date": "2024-12-31",
	"daily": ["precipitation_sum", "cloud_cover_mean"],
	"timezone": "auto",
}
responses = openmeteo.weather_api(url, params=params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process daily data. The order of variables needs to be the same as requested.
daily = response.Daily()
daily_precipitation_sum = daily.Variables(0).ValuesAsNumpy()
daily_cloud_cover_mean = daily.Variables(1).ValuesAsNumpy()

daily_data = {"date": pd.date_range(
	start = pd.to_datetime(daily.Time() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	end =  pd.to_datetime(daily.TimeEnd() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = daily.Interval()),
	inclusive = "left"
)}

daily_data["precipitation_sum"] = daily_precipitation_sum
daily_data["cloud_cover_mean"] = daily_cloud_cover_mean

daily_dataframe = pd.DataFrame(data = daily_data)

print("\nDaily data\n", daily_dataframe)

daily_dataframe.to_csv("daily_weather_data.csv", index = False)

Coordinates: 52.5483283996582°N 13.407821655273438°E
Elevation: 38.0 m asl
Timezone: b'Europe/Berlin'b'GMT+1'
Timezone difference to GMT+0: 3600s

Daily data
                          date  precipitation_sum  cloud_cover_mean
0   2024-01-01 00:00:00+00:00           1.900000         72.833336
1   2024-01-02 00:00:00+00:00           8.500000         99.416664
2   2024-01-03 00:00:00+00:00          10.800000         85.375000
3   2024-01-04 00:00:00+00:00           2.800000         93.083336
4   2024-01-05 00:00:00+00:00           5.400001         97.500000
..                        ...                ...               ...
361 2024-12-27 00:00:00+00:00           0.000000         96.250000
362 2024-12-28 00:00:00+00:00           0.000000         89.208336
363 2024-12-29 00:00:00+00:00           0.000000         66.291664
364 2024-12-30 00:00:00+00:00           0.000000         95.708336
365 2024-12-31 00:00:00+00:00           0.000000         66.791664

[366 rows x 3 columns]


In [4]:
# save data as csv file


df = daily_dataframe.copy()  # Create a copy to work with, preserving the original data

# 1. Classification Logic (Open-Meteo columns)
def classify_weather(row):
    if row['precipitation_sum'] > 0:
        return 'Rainy'
    elif row['cloud_cover_mean'] > 75:
        return 'Cloudy'
    else:
        return 'Sunny'

# Assuming 'df' is your existing Open-Meteo dataframe
df['state'] = df.apply(classify_weather, axis=1)

# 2. Create the "Tomorrow" column by shifting the data
df['tomorrow_state'] = df['state'].shift(-1)

# 3. Create the Transition Matrix
# normalize=0 (or 'index') makes each row sum to 1 (probabilities)
transition_matrix = pd.crosstab(
    df['state'], 
    df['tomorrow_state'], 
    normalize='index'
)

print("--- Weather Transition Matrix (Probabilities) ---")
print(transition_matrix)

# Optional: View the raw counts to see how much data is backing each probability
counts = pd.crosstab(df['state'], df['tomorrow_state'])
print("\n--- Transition Counts ---")
print(counts)

--- Weather Transition Matrix (Probabilities) ---
tomorrow_state    Cloudy     Rainy     Sunny
state                                       
Cloudy          0.313725  0.431373  0.254902
Rainy           0.087558  0.741935  0.170507
Sunny           0.164948  0.340206  0.494845

--- Transition Counts ---
tomorrow_state  Cloudy  Rainy  Sunny
state                               
Cloudy              16     22     13
Rainy               19    161     37
Sunny               16     33     48


In [5]:
import numpy as np

# Convert the transition matrix DataFrame to a numpy array.
# States are in alphabetical order: Cloudy, Rainy, Sunny
states = list(transition_matrix.index)
P = transition_matrix.values
print("States:", states)
print("Transition matrix P:\n", P)

# ── Q1: "Given today is Sunny, what's the chance it'll Rain in two days?" ──
# Compute P^2 (the 2-step transition matrix) via matrix multiplication.
# Entry (i, j) of P^n = probability of going from state i to state j in n steps.
P2 = np.linalg.matrix_power(P, 2)
P2_df = pd.DataFrame(P2, index=states, columns=states)
print("\n=== 2-Step Transition Matrix P² ===")
print(P2_df.round(4))

sunny_idx = states.index('Sunny')
rainy_idx = states.index('Rainy')
print(f"\nP(Rain in 2 days | Sunny today) = {P2[sunny_idx, rainy_idx]:.4f}")

# ── Q2: "Long-run stationary distribution" ──
# The stationary distribution π satisfies π·P = π and sum(π) = 1.
# We find it as the left eigenvector of P with eigenvalue 1.
eigenvalues, left_eigenvectors = np.linalg.eig(P.T)

# Find the eigenvector corresponding to eigenvalue ≈ 1
unit_idx = np.argmin(np.abs(eigenvalues - 1.0))
stationary = np.real(left_eigenvectors[:, unit_idx])
stationary = stationary / stationary.sum()  # normalize to sum to 1

print("\n=== Stationary (Long-Run) Distribution ===")
for s, prob in zip(states, stationary):
    print(f"  π({s}) = {prob:.4f}")

# Answer the practical question: rain tents per 100 event-days
rainy_frac = stationary[rainy_idx]
print(f"\nLong-run fraction of Rainy days: {rainy_frac:.4f}")
print(f"Rain tents needed per 100 event-days: {rainy_frac * 100:.1f}")

# ── Q3: "Mean return time to Sunny" ──
# For an ergodic Markov chain, the mean return time to state i = 1 / π(i).
# This is the expected number of days between visits to that state.
print("\n=== Mean Return Times ===")
for s, prob in zip(states, stationary):
    mean_return = 1.0 / prob
    print(f"  E[return to {s}] = {mean_return:.2f} days")

sunny_return = 1.0 / stationary[sunny_idx]
print(f"\nOn average, a Sunny day occurs every {sunny_return:.2f} days — useful for scheduling!")

States: ['Cloudy', 'Rainy', 'Sunny']
Transition matrix P:
 [[0.31372549 0.43137255 0.25490196]
 [0.0875576  0.74193548 0.17050691]
 [0.16494845 0.34020619 0.49484536]]

=== 2-Step Transition Matrix P² ===
        Cloudy   Rainy   Sunny
Cloudy  0.1782  0.5421  0.2797
Rainy   0.1206  0.6462  0.2332
Sunny   0.1632  0.4919  0.3449

P(Rain in 2 days | Sunny today) = 0.4919

=== Stationary (Long-Run) Distribution ===
  π(Cloudy) = 0.1401
  π(Rainy) = 0.5900
  π(Sunny) = 0.2699

Long-run fraction of Rainy days: 0.5900
Rain tents needed per 100 event-days: 59.0

=== Mean Return Times ===
  E[return to Cloudy] = 7.14 days
  E[return to Rainy] = 1.69 days
  E[return to Sunny] = 3.71 days

On average, a Sunny day occurs every 3.71 days — useful for scheduling!
